# CommonsenseQA — CoT Baseline (Direct Comparison Version)

**Purpose:** Produce a 3-way comparison where the baseline in this notebook matches the
original guided run (70.7%). Previous CoT notebook used a different prompt format that
caused the baseline to collapse to 39.78%.

**Fix:** This notebook uses the same simple, natural MCQ prompt as the original guided
notebook baseline. Cell 10 runs a 50-question verification before the full run.

| Condition | Compute | Expected |
|---|---|---|
| Baseline (this notebook) | 7.5B | ~70.7% ← must match |
| CoT (this notebook) | 7.5B | TBD |
| Guided (original run) | 10.5B | 75.8% |

> N=900 · Seed=42 · Same questions as original guided run

In [1]:
# CELL 1 -- Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# CELL 2 -- Login
from huggingface_hub import login
login("")  # paste token
print("Login done")

Login done


In [3]:
# CELL 3 -- Imports
import os, json, re, random, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/csqa_cot_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Output: {OUTPUT_DIR}")

GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB
Output: /kaggle/working/csqa_cot_v2


In [4]:
# CELL 4 -- Config
# CRITICAL: N=900 and seed=42 MUST match original guided CSQA run.
CONFIG = {
    "solver_model"      : "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_name"      : "tau/commonsense_qa",
    "dataset_split"     : "validation",
    "max_eval_samples"  : 900,
    "random_seed"       : 42,
    "n_votes"           : 5,
    "vote_temperature"  : 0.4,
    "max_new_tokens"    : 200,
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "angle1_file"       : f"{OUTPUT_DIR}/angle1_compute.json",
    "angle2_file"       : f"{OUTPUT_DIR}/angle2_consistency.json",
    "angle3_file"       : f"{OUTPUT_DIR}/angle3_calibration.json",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"        : 50,
}
print("Config ready")

Config ready


In [5]:
# CELL 5 -- Load CSQA
print("Loading CommonsenseQA...")
raw_ds = load_dataset(CONFIG["dataset_name"])

def normalise_csqa(item):
    q = item["question"].strip()
    choices = list(zip(item["choices"]["label"], item["choices"]["text"]))
    choice_str = "\n".join(f"{lbl}. {txt}" for lbl, txt in choices)
    full_q = f"{q}\n\n{choice_str}"
    return {"question": full_q, "answer": item["answerKey"].strip().upper()}

all_data = [normalise_csqa(x) for x in raw_ds[CONFIG["dataset_split"]]]
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
else:
    test_data = all_data

ans_dist = Counter(d["answer"] for d in test_data)
print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
for opt in ['A','B','C','D','E']:
    n = ans_dist.get(opt,0)
    print(f"  {opt}: {n} ({n/len(test_data)*100:.1f}%)")

Loading CommonsenseQA...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

Sampled 900 questions (seed=42)
  A: 170 (18.9%)
  B: 194 (21.6%)
  C: 166 (18.4%)
  D: 190 (21.1%)
  E: 180 (20.0%)


In [6]:
# CELL 6 -- Answer extraction
def extract_mcq_answer(text):
    text = text.strip()
    # 1. Explicit statement
    m = re.search(r"(?:the answer is|answer is|answer:|correct answer is|answer =)\s*\**([A-E])\**", text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # 2. Therefore/so/thus
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\**([A-E])\**", text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # 3. Last non-empty line
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if lines:
        m = re.match(r"^\**([A-E])\**[.):,]?$", lines[-1], re.IGNORECASE)
        if m: return m.group(1).upper()
    # 4. Bold letter at end
    m = re.search(r"\*\*([A-E])\*\*\.?\s*$", text)
    if m: return m.group(1).upper()
    # 5. Parenthesised letter at end
    m = re.search(r"\(([A-E])\)\s*$", text)
    if m: return m.group(1).upper()
    # 6. Any A-E at end of last line
    m = re.search(r"\b([A-E])\b[.\s]*$", text)
    if m: return m.group(1).upper()
    # 7. First standalone letter anywhere
    m = re.search(r"^([A-E])[\s.)]", text, re.MULTILINE)
    if m: return m.group(1).upper()
    return ""

# Tests
_t = [('The answer is A','A'),('answer: B','B'),('Therefore C','C'),
      ('\nD','D'),('**E**','E'),('(A)','A'),('none','')]
ok = all(extract_mcq_answer(t)==e for t,e in _t)
print("Extractor:", "ALL PASSED" if ok else "FAIL")

Extractor: ALL PASSED


In [7]:
# CELL 7 -- Load solver
print(f"Loading: {CONFIG['solver_model']}")
solver_tok = AutoTokenizer.from_pretrained(CONFIG["solver_model"])
if solver_tok.pad_token is None: solver_tok.pad_token = solver_tok.eos_token

solver_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["solver_model"], torch_dtype=torch.float16, device_map="auto"
).eval()
vram = torch.cuda.memory_allocated()/1e9
print(f"VRAM: {vram:.2f}GB | Headroom: {15.0-vram:.1f}GB")

Loading: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM: 3.09GB | Headroom: 11.9GB


In [8]:
# CELL 8 -- Prompts
#
# BASELINE_SYSTEM: Simple natural prompt — matches the original guided notebook baseline.
# This is what produced 70.7% in the original run.
# No rigid format constraint — just asks for the answer letter naturally.
#
# COT_SYSTEM: Same prompt + 'Let's think step by step'

BASELINE_SYSTEM = (
    "You are a helpful assistant answering multiple choice questions.\n"
    "Read the question and all options carefully.\n"
    "Respond with just the letter of the correct answer: A, B, C, D, or E."
)

COT_SYSTEM = (
    "You are a helpful assistant answering multiple choice questions.\n"
    "Read the question and all options carefully.\n"
    "Let's think step by step.\n"
    "After reasoning, respond with just the letter of the correct answer: A, B, C, D, or E."
)

REFINER_SYSTEM = (
    "You are a careful reasoning assistant.\n"
    "Previous attempts gave conflicting answers. Read the question from scratch.\n"
    "Respond with just the letter of the correct answer: A, B, C, D, or E."
)

def run_solver(messages, temperature):
    prompt = solver_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = solver_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(solver_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = solver_model.generate(
            **inputs, max_new_tokens=CONFIG["max_new_tokens"],
            temperature=max(temperature, 0.05), do_sample=True,
            top_p=0.92, top_k=40,
            pad_token_id=solver_tok.eos_token_id, repetition_penalty=1.1,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return solver_tok.decode(new_toks, skip_special_tokens=True).strip()

def generate_baseline(q):
    return run_solver([{"role":"system","content":BASELINE_SYSTEM},
                       {"role":"user","content":f"Question:\n{q}"}], CONFIG["vote_temperature"])

def generate_cot(q):
    return run_solver([{"role":"system","content":COT_SYSTEM},
                       {"role":"user","content":f"Question:\n{q}"}], CONFIG["vote_temperature"])

def generate_refiner(q, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = f"Question:\n{q}\n\nPrevious attempts: {cands}\nDecide:"
    return run_solver([{"role":"system","content":REFINER_SYSTEM},
                       {"role":"user","content":content}], 0.3)

print("Prompts ready. BASELINE_SYSTEM matches original guided notebook baseline.")

Prompts ready. BASELINE_SYSTEM matches original guided notebook baseline.


In [9]:
# CELL 9 -- Voting logic
def vote_and_decide(answers, question, gt_answer=None):
    valid = [a for a in answers if a and a.strip()] or answers
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer = most_common[0][0]; top_count = most_common[0][1]; total = len(valid)
    correct_votes = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority = (len(most_common)==1 or top_count > most_common[1][1])
    refiner_used = False; refiner_correct = None
    if is_majority:
        final=top_answer; strategy="majority"; conf=round(top_count/total,4); wasted=total-top_count
    else:
        ref_raw = generate_refiner(question, list(valid))
        ref_ans = extract_mcq_answer(ref_raw)
        refiner_used=True; refiner_correct=(ref_ans==gt_answer) if gt_answer else None
        all_v = valid+([ref_ans] if ref_ans else [])
        new_counts=Counter(all_v); new_common=new_counts.most_common()
        new_top=new_common[0][0]; new_top_c=new_common[0][1]
        still_tied=len(new_common)>1 and new_top_c==new_common[1][1]
        final=new_top; strategy="coin_flip" if still_tied else "refiner_tiebreak"
        conf=round(new_top_c/len(all_v),4); total=len(all_v)
        correct_votes=Counter(all_v).get(gt_answer,0) if gt_answer else 0
        vote_consistency=correct_votes/max(total,1); wasted=total-new_top_c
        vote_counts=new_counts
    return {"final_answer":final,"strategy":strategy,"confidence":conf,
            "vote_counts":dict(vote_counts),"correct_votes":correct_votes,
            "total_votes":total,"vote_consistency":round(vote_consistency,4),
            "wasted_votes":wasted,"refiner_used":refiner_used,"refiner_correct":refiner_correct}

print("Voting logic ready")

Voting logic ready


In [10]:
# CELL 10 -- VERIFICATION RUN (50 questions)
# Run baseline on first 50 questions.
# Expected: ~70% accuracy. If it's below 60%, the prompt is wrong — do not continue.
# If it's 65-75%, proceed to Cell 11.

verify_n = 50
verify_correct = 0; verify_empty = 0

print(f"Verification run: {verify_n} questions...")
for i in range(verify_n):
    item = test_data[i]; q = item["question"]; gt = item["answer"]
    votes = [extract_mcq_answer(generate_baseline(q)) for _ in range(CONFIG["n_votes"])]
    empty = sum(1 for v in votes if not v)
    verify_empty += empty
    dec = vote_and_decide(votes, q, gt)
    if dec["final_answer"] == gt: verify_correct += 1

v_acc = verify_correct / verify_n * 100
v_empty_rate = verify_empty / (verify_n * CONFIG["n_votes"]) * 100
print(f"\n=== VERIFICATION RESULT ===")
print(f"  Baseline accuracy : {v_acc:.1f}%")
print(f"  Empty answer rate  : {v_empty_rate:.1f}%")
print(f"  Target             : ~70%")
print()
if v_acc >= 60:
    print(f"PASS — Baseline looks correct. Proceed to Cell 11.")
elif v_acc >= 50:
    print(f"WARNING — Baseline lower than expected ({v_acc:.1f}%). Check prompt before proceeding.")
else:
    print(f"FAIL — Baseline collapsed ({v_acc:.1f}%). Do NOT run Cell 11. Check the prompt in Cell 8.")

Verification run: 50 questions...

=== VERIFICATION RESULT ===
  Baseline accuracy : 70.0%
  Empty answer rate  : 0.0%
  Target             : ~70%

PASS — Baseline looks correct. Proceed to Cell 11.


In [11]:
# CELL 11 -- Full Dual Evaluation (N=900)
# Only run this if Cell 10 verification passed (baseline ≈ 70%).
# Estimated time: ~3-4 hours on T4. Checkpoints every 50 questions.

print(f"Full evaluation: {len(test_data)} questions, 2 conditions each")
print(f"Estimated time: ~3-4 hours on T4")
print("-"*65)

cot_results=[]; base_results=[]; start_idx=0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f: ckpt=json.load(f)
    start_idx=ckpt.get("last_index",0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines=[json.loads(l) for l in f if l.strip()]
        cot_results=[r for r in lines if r.get("mode")=="cot"]
        base_results=[r for r in lines if r.get("mode")=="baseline"]
    print(f"Resumed from {start_idx} (CoT:{len(cot_results)} Base:{len(base_results)})")
else:
    print("Starting fresh")

t0=time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="CSQA CoT v2"):
    item=test_data[idx]; question=item["question"]; gt_answer=item["answer"]

    # Baseline
    try:
        b_votes=[extract_mcq_answer(generate_baseline(question)) for _ in range(CONFIG["n_votes"])]
        b_dec=vote_and_decide(b_votes,question,gt_answer)
        base_results.append({
            "mode":"baseline","idx":idx,"question":question,"gt_answer":gt_answer,
            "final_answer":b_dec["final_answer"],"correct":b_dec["final_answer"]==gt_answer,
            "strategy":b_dec["strategy"],"confidence":b_dec["confidence"],
            "correct_votes":b_dec["correct_votes"],"total_votes":b_dec["total_votes"],
            "vote_consistency":b_dec["vote_consistency"],"wasted_votes":b_dec["wasted_votes"],
            "refiner_used":False,"refiner_correct":None,"vote_counts":b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode":"baseline","idx":idx,"question":question,"gt_answer":gt_answer,
            "final_answer":"","correct":False,"strategy":"error","confidence":0.0,
            "correct_votes":0,"total_votes":5,"vote_consistency":0.0,"wasted_votes":5,
            "refiner_used":False,"refiner_correct":None,"vote_counts":{},"error":str(e),
        })

    # CoT
    try:
        c_votes=[extract_mcq_answer(generate_cot(question)) for _ in range(CONFIG["n_votes"])]
        c_dec=vote_and_decide(c_votes,question,gt_answer)
        cot_results.append({
            "mode":"cot","idx":idx,"question":question,"gt_answer":gt_answer,
            "final_answer":c_dec["final_answer"],"correct":c_dec["final_answer"]==gt_answer,
            "strategy":c_dec["strategy"],"confidence":c_dec["confidence"],
            "correct_votes":c_dec["correct_votes"],"total_votes":c_dec["total_votes"],
            "vote_consistency":c_dec["vote_consistency"],"wasted_votes":c_dec["wasted_votes"],
            "refiner_used":c_dec["refiner_used"],"refiner_correct":c_dec["refiner_correct"],
            "vote_counts":c_dec["vote_counts"],
        })
    except RuntimeError as e:
        cot_results.append({
            "mode":"cot","idx":idx,"question":question,"gt_answer":gt_answer,
            "final_answer":"","correct":False,"strategy":"error","confidence":0.0,
            "correct_votes":0,"total_votes":5,"vote_consistency":0.0,"wasted_votes":5,
            "refiner_used":False,"refiner_correct":None,"vote_counts":{},"error":str(e),
        })

    if (idx+1) % CONFIG["save_every"]==0:
        with open(CONFIG["results_file"],"w") as f:
            for r in cot_results+base_results: f.write(json.dumps(r)+"\n")
        with open(CONFIG["checkpoint_file"],"w") as f: json.dump({"last_index":idx+1},f)
        b_a=sum(r["correct"] for r in base_results)/len(base_results)*100
        c_a=sum(r["correct"] for r in cot_results)/len(cot_results)*100
        mins=(time.time()-t0)/60
        print(f"  [{idx+1:4d}] Base:{b_a:.1f}%  CoT:{c_a:.1f}%  ({mins:.1f}min)")

with open(CONFIG["results_file"],"w") as f:
    for r in cot_results+base_results: f.write(json.dumps(r)+"\n")

b_c=sum(r["correct"] for r in base_results); c_c=sum(r["correct"] for r in cot_results)
print(f"\nDone.")
print(f"  Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%  (target: 70.7%)")
print(f"  CoT      : {c_c}/{len(cot_results)} = {c_c/len(cot_results)*100:.1f}%")
print(f"  Guided   : 75.8% (from original run)")   

Full evaluation: 900 questions, 2 conditions each
Estimated time: ~3-4 hours on T4
-----------------------------------------------------------------
Starting fresh


CSQA CoT v2:   0%|          | 0/900 [00:00<?, ?it/s]

  [  50] Base:70.0%  CoT:62.0%  (4.2min)
  [ 100] Base:69.0%  CoT:65.0%  (9.9min)
  [ 150] Base:72.7%  CoT:69.3%  (14.5min)
  [ 200] Base:73.0%  CoT:70.0%  (20.0min)
  [ 250] Base:72.8%  CoT:71.2%  (23.9min)
  [ 300] Base:72.7%  CoT:70.3%  (28.7min)
  [ 350] Base:72.0%  CoT:67.7%  (34.5min)
  [ 400] Base:73.0%  CoT:69.8%  (38.7min)
  [ 450] Base:72.9%  CoT:69.8%  (42.9min)
  [ 500] Base:71.8%  CoT:68.6%  (48.4min)
  [ 550] Base:70.7%  CoT:67.8%  (54.8min)
  [ 600] Base:71.0%  CoT:67.7%  (60.9min)
  [ 650] Base:70.6%  CoT:67.4%  (65.1min)
  [ 700] Base:71.3%  CoT:68.3%  (69.5min)
  [ 750] Base:71.2%  CoT:68.4%  (73.9min)
  [ 800] Base:70.5%  CoT:67.6%  (79.9min)
  [ 850] Base:70.9%  CoT:68.0%  (84.1min)
  [ 900] Base:70.4%  CoT:67.7%  (88.4min)

Done.
  Baseline : 634/900 = 70.4%  (target: 70.7%)
  CoT      : 609/900 = 67.7%
  Guided   : 75.8% (from original run)


In [12]:
# CELL 12 -- ANGLE 1: Compute Efficiency
GUIDED_ACC = 75.8

b_acc = sum(r["correct"] for r in base_results)/len(base_results)*100
c_acc = sum(r["correct"] for r in cot_results)/len(cot_results)*100
b_empty = sum(1 for r in base_results if not r.get('final_answer','').strip())
c_empty = sum(1 for r in cot_results  if not r.get('final_answer','').strip())

print("="*65)
print("ANGLE 1 -- COMPUTE EFFICIENCY")
print("="*65)
print(f"  Baseline  (7.5B)  : {b_acc:.2f}%  (target: 70.7%  match: {'✓' if abs(b_acc-70.7)<3 else '✗ MISMATCH'})")
print(f"  CoT       (7.5B)  : {c_acc:.2f}%")
print(f"  Guided    (10.5B) : {GUIDED_ACC}%")
print(f"  CoT vs Baseline   : {c_acc-b_acc:+.1f} pts")
print(f"  Guided vs CoT     : {GUIDED_ACC-c_acc:+.1f} pts")
print(f"  Guided vs Baseline: {GUIDED_ACC-b_acc:+.1f} pts")
print(f"  Empty (base/CoT)  : {b_empty} / {c_empty}")

angle1={
    "dataset":"CommonsenseQA","n":len(base_results),"compute_B":7.5,
    "baseline_accuracy":round(b_acc,2),"cot_accuracy":round(c_acc,2),
    "guided_accuracy_known":GUIDED_ACC,
    "cot_vs_baseline":round(c_acc-b_acc,2),"guided_vs_cot":round(GUIDED_ACC-c_acc,2),
    "baseline_empty":b_empty,"cot_empty":c_empty,
}
with open(CONFIG["angle1_file"],"w") as f: json.dump(angle1,f,indent=2)
print(f"Saved -> {CONFIG['angle1_file']}")

ANGLE 1 -- COMPUTE EFFICIENCY
  Baseline  (7.5B)  : 70.44%  (target: 70.7%  match: ✓)
  CoT       (7.5B)  : 67.67%
  Guided    (10.5B) : 75.8%
  CoT vs Baseline   : -2.8 pts
  Guided vs CoT     : +8.1 pts
  Guided vs Baseline: +5.4 pts
  Empty (base/CoT)  : 0 / 0
Saved -> /kaggle/working/csqa_cot_v2/angle1_compute.json


In [13]:
# CELL 13 -- ANGLE 2: Vote Consistency
import numpy as np

b_cons=[r["vote_consistency"] for r in base_results]
c_cons=[r["vote_consistency"] for r in cot_results]
b_mean=np.mean(b_cons); c_mean=np.mean(c_cons)
lift=c_mean/max(b_mean,1e-6)
cot_wins=sum(1 for a,b in zip(c_cons,b_cons) if a>b)
base_wins=sum(1 for a,b in zip(c_cons,b_cons) if b>a)
tied=sum(1 for a,b in zip(c_cons,b_cons) if a==b)

def bucket(scores):
    return {
        "all_wrong (0%)":sum(1 for s in scores if s==0.0),
        "low (1-39%)":sum(1 for s in scores if 0<s<0.4),
        "medium (40-79%)":sum(1 for s in scores if 0.4<=s<0.8),
        "high (80-100%)":sum(1 for s in scores if s>=0.8),
    }

print(f"  CoT  mean consistency : {c_mean*100:.1f}%")
print(f"  Base mean consistency : {b_mean*100:.1f}%")
print(f"  Lift                  : {lift:.2f}x")
print(f"  CoT wins / Base wins / Tied: {cot_wins}/{base_wins}/{tied}")

cot_opts=Counter(r["final_answer"] for r in cot_results)
b_opts=Counter(r["final_answer"] for r in base_results)
print("\n  Option dist (CoT):  ", {o:cot_opts.get(o,0) for o in list('ABCDE')+['']})
print("  Option dist (Base): ", {o:b_opts.get(o,0) for o in list('ABCDE')+['']})

angle2={
    "cot_mean_consistency":round(c_mean,4),"baseline_mean_consistency":round(b_mean,4),
    "consistency_lift":round(lift,4),"cot_wins":cot_wins,"baseline_wins":base_wins,"tied":tied,
    "cot_distribution":bucket(c_cons),"baseline_distribution":bucket(b_cons),
    "cot_option_dist":dict(cot_opts),"baseline_option_dist":dict(b_opts),
}
with open(CONFIG["angle2_file"],"w") as f: json.dump(angle2,f,indent=2)
print(f"Saved -> {CONFIG['angle2_file']}")

  CoT  mean consistency : 66.2%
  Base mean consistency : 70.3%
  Lift                  : 0.94x
  CoT wins / Base wins / Tied: 86/164/650

  Option dist (CoT):   {'A': 131, 'B': 207, 'C': 133, 'D': 154, 'E': 275, '': 0}
  Option dist (Base):  {'A': 184, 'B': 218, 'C': 149, 'D': 160, 'E': 189, '': 0}
Saved -> /kaggle/working/csqa_cot_v2/angle2_consistency.json


In [14]:
# CELL 14 -- ANGLE 3: Calibration

def ece_report(results, label):
    buckets=[(lambda c:c>=0.80,0.90),(lambda c:0.60<=c<0.80,0.70),
             (lambda c:0.40<=c<0.60,0.50),(lambda c:c<0.40,0.25)]
    n_total=len(results); ece=0.0; calib=[]
    false_conf=sum(1 for r in results if r["confidence"]>=0.80 and not r["correct"])
    for cond,mid in buckets:
        sub=[r for r in results if cond(r["confidence"])]
        if not sub: continue
        acc=sum(r["correct"] for r in sub)/len(sub)
        gap=abs(acc-mid); ece+=(len(sub)/n_total)*gap
        calib.append({"bucket":str(mid),"count":len(sub),"accuracy":round(acc,4),"gap":round(gap,4)})
    print(f"  [{label}] ECE={ece:.4f}  false_conf={false_conf}")
    return ece, calib, false_conf

print("="*65)
print("ANGLE 3 -- CALIBRATION")
print("="*65)
c_ece,c_calib,c_false=ece_report(cot_results,"CoT")
b_ece,b_calib,b_false=ece_report(base_results,"Baseline")
guided_ece=0.141
imp=(b_ece-c_ece)/max(b_ece,1e-6)*100
print(f"\n  ECE ladder: Base={b_ece:.4f} → CoT={c_ece:.4f} → Guided={guided_ece}")
print(f"  CoT vs Base imp: {imp:.1f}%")
print(f"  Guided vs CoT imp: {(c_ece-guided_ece)/c_ece*100:.1f}%")

angle3={
    "cot_ece":round(c_ece,4),"baseline_ece":round(b_ece,4),"guided_ece_known":guided_ece,
    "ece_improvement_pct":round(imp,2),"cot_false_conf":c_false,"baseline_false_conf":b_false,
    "cot_calibration":c_calib,"baseline_calibration":b_calib,
}
with open(CONFIG["angle3_file"],"w") as f: json.dump(angle3,f,indent=2)
print(f"Saved -> {CONFIG['angle3_file']}")

ANGLE 3 -- CALIBRATION
  [CoT] ECE=0.1908  false_conf=208
  [Baseline] ECE=0.1841  false_conf=234

  ECE ladder: Base=0.1841 → CoT=0.1908 → Guided=0.141
  CoT vs Base imp: -3.6%
  Guided vs CoT imp: 26.1%
Saved -> /kaggle/working/csqa_cot_v2/angle3_calibration.json


In [15]:
# CELL 15 -- Final 3-Way Summary

b_acc = sum(r["correct"] for r in base_results)/len(base_results)*100
c_acc = sum(r["correct"] for r in cot_results)/len(cot_results)*100
GUIDED=75.8

print("="*70)
print("COMMONSENSE QA — 3-WAY COMPARISON (Direct Comparison)")
print("="*70)
print(f"  N={len(base_results)}  Seed=42  Same questions as original guided run")
print()
print(f"  Condition           | Compute | Accuracy | vs Baseline")
print(f"  --------------------|---------|----------|------------")
print(f"  Baseline (no guide) |   7.5B  | {b_acc:>7.2f}% | —")
print(f"  CoT                 |   7.5B  | {c_acc:>7.2f}% | {c_acc-b_acc:>+9.1f} pts")
print(f"  Guided (original)   |  10.5B  | {GUIDED:>7.1f}% | {GUIDED-b_acc:>+9.1f} pts")
print()
print(f"  Guide adds over CoT : {GUIDED-c_acc:+.1f} pts")
print(f"  CoT captures        : {(c_acc-b_acc)/(GUIDED-b_acc)*100:.1f}% of guided gain")
print()
print(f"  Baseline match check: {b_acc:.1f}% vs expected 70.7% — {'MATCH ✓' if abs(b_acc-70.7)<3 else 'MISMATCH ✗'}")

COMMONSENSE QA — 3-WAY COMPARISON (Direct Comparison)
  N=900  Seed=42  Same questions as original guided run

  Condition           | Compute | Accuracy | vs Baseline
  --------------------|---------|----------|------------
  Baseline (no guide) |   7.5B  |   70.44% | —
  CoT                 |   7.5B  |   67.67% |      -2.8 pts
  Guided (original)   |  10.5B  |    75.8% |      +5.4 pts

  Guide adds over CoT : +8.1 pts
  CoT captures        : -51.9% of guided gain

  Baseline match check: 70.4% vs expected 70.7% — MATCH ✓
